In [4]:
import os
from dotenv import load_dotenv
load_dotenv()
KEY: str = os.environ["KOREAN_DICT_KEY"]
print(KEY[:4] + "***")
# 키 전체가 출력되지 않도록 앞 4자만 확인

2DE5***


In [1]:
%pip install python-dotenv requests


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
import requests

def search_word(q: str, num: int = 10, start: int = 1) -> dict:
    url = "https://opendict.korean.go.kr/api/search"
    params = {
        "key": KEY,
        "q": q,
        "req_type": "json",
        "num": num,
        "start": start,
        "type1": "word",
    }
    r = requests.get(url, params=params, timeout=10)
    r.raise_for_status()
    return r.json()


In [6]:
import json

data = search_word("김치")
print(json.dumps(data, ensure_ascii=False, indent=2)[:400])

{
  "channel": {
    "total": 328,
    "num": 10,
    "title": "우리말샘 개발 지원(Open API) - 사전 어휘 검색",
    "start": 1,
    "description": "우리말샘 개발 지원(Open API) - 사전 어휘 검색 결과",
    "link": "https://opendict.korean.go.kr",
    "item": [
      {
        "word": "김치",
        "sense": [
          {
            "syntacticArgument": "",
            "syntacticAnnotation": "",
            "cat": "",
          


In [8]:
channel = data["channel"]
total = channel["total"]
items = channel["item"]
n = len(items)

print(f"총 {total}건, 이 페이지 {n}건")

for item in items[:5]:
    word = item["word"]
    pos = item.get("pos", "품사 없음")
    sense = item["sense"]
    if isinstance(sense, list):
        definition = sense[0]["definition"]
    else:
        definition = sense["definition"]
    print(f"{word} ({pos}) --> {definition[:40]}")


총 328건, 이 페이지 10건
김치 (품사 없음) --> 소금에 절인 배추나 무 따위를 고춧가루, 파, 마늘 따위의 양념에 버무린
김-치 (품사 없음) --> 고려 말기·조선 초기의 문신(?~?). 자는 기보(基甫). 김해 부사를 
김-치 (품사 없음) --> 조선 중기의 문신(1577~1625). 자는 사정(士精). 호는 남봉(南
김치 공장 (품사 없음) --> 김치를 만드는 공장.
김치 보릿고개 (품사 없음) --> 김장철인 가을·겨울과 달리 상대적으로 김치가 부족한 봄여름을 비유적으로 


search_word는 우리말샘 검색 엔드포인트를 requests.get으로 호출한다.
timeout=10으로 응답 지연을 막고 raise_for_status()로 HTTP 오류를 검사한 뒤 r.json()을 반환
json.dumps로 보기 좋게 출력. ensure_ascii=False를 빼면 한글이 \ㅕ___ 형태의 유니코드 이스케이프로 출력되어 읽기 어려워짐
응답은 data["channel"] 아래에 전체 결과 수 total과 항목 리스트 item이 들어있고 pos 필드는 모든 항목에 있지는 않으므로 dict.get("pos", "품사 없음")으로 안전하게 접근해 표제어/품사/뜻풀이 앞 40자를 출력한다.

In [12]:
import time

words: list[str] = [
    "김치", "라면", "만두", "김밥", "국수", "떡볶이", "불고기", "비빔밥",
]

results: dict[str, dict] = {}
for w in words:
    data = search_word(w)
    results[w] = data
    print(f"{w}: {data['channel']['total']}건")
    time.sleep(0.3)

김치: 328건
라면: 86건
만두: 89건
김밥: 39건
국수: 227건
떡볶이: 24건
불고기: 38건


ConnectTimeout: HTTPSConnectionPool(host='opendict.korean.go.kr', port=443): Max retries exceeded with url: /api/search?key=2DE5644640D0F63B11D7D28F7C58D3A4&q=%EB%B9%84%EB%B9%94%EB%B0%A5&req_type=json&num=10&start=1&type1=word (Caused by ConnectTimeoutError(<HTTPSConnection(host='opendict.korean.go.kr', port=443) at 0x7372589a1d60>, 'Connection to opendict.korean.go.kr timed out. (connect timeout=10)'))

#gpt (원인을 몰라서 사용)

에러 자체는 코드 문제가 아니라 네트워크 연결이 안 돼서 생긴 거예요. ConnectTimeout은 서버(opendict.korean.go.kr)에 10초 안에 접속조차 못 했다는 뜻입니다. 코드가 틀린 게 아니라, 요청을 보냈는데 응답이 오지 않은 상황이에요.

가능한 원인이 몇 가지 있습니다.

가장 흔한 건 실행 환경에 외부 인터넷이 차단된 경우예요. 학교 실습 서버나 일부 클라우드 노트북(특정 JupyterHub 등)은 외부 사이트 접속을 막아둡니다. 이걸 먼저 확인해보세요:

Copyimport requests
r = requests.get("https://opendict.korean.go.kr", timeout=10)
print(r.status_code)
이것도 타임아웃이 나면 환경 자체가 외부 접속을 막고 있는 거예요. 그렇다면 인터넷이 되는 환경(예: 로컬 주피터, Google Colab)에서 실행해야 합니다.

다른 가능성으로는 그 순간 우리말샘 서버가 일시적으로 느렸거나 잠깐 다운됐을 수 있어요. 이 경우 잠시 뒤 다시 실행하면 됩니다. 또는 학교/회사 방화벽이나 프록시가 막고 있을 수도 있습니다.

확인 순서를 정리하면 이렇습니다.

키 재발급 (노출됐으므로)
위의 간단한 접속 테스트 코드 실행해서 환경이 외부 접속을 막는지 확인
막혀있다면 Colab이나 로컬에서 실행, 일시적 문제 같으면 잠시 후 재시도
혹시 지금 코드를 어디서 실행하고 계신가요? (학교 서버 / Colab / 본인 노트북 등) 알려주시면 그 환경에 맞게 더 구체적으로 도와드릴게요.

In [14]:
import time

words: list[str] = [
    "김치", "라면", "만두", "김밥", "국수", "떡볶이", "불고기",
]

results: dict[str, dict] = {}
for w in words:
    data = search_word(w)
    results[w] = data
    print(f"{w}: {data['channel']['total']}건")
    time.sleep(0.3)

김치: 328건
라면: 86건
만두: 89건
김밥: 39건
국수: 227건
떡볶이: 24건
불고기: 38건


In [16]:
from collections import Counter

def get_pos(item: dict) -> str:
    pos = item.get("pos")
    if not pos:
        sense = item.get("sense")
        if isinstance(sense, list) and sense:
            pos = sense[0].get("pos")
        elif isinstance(sense, dict):
            pos = sense.get("pos")
    return pos or "(미상)"

all_items: list[dict] = []
for w in words:
    all_items.extend(results[w]["channel"]["item"])

counter: Counter[str] = Counter(get_pos(item) for item in all_items)

for pos, freq in counter.most_common(3):
    print(f"{pos}: {freq}")


명사: 50
(미상): 19
어미: 1


for문으로 8개 검색어를 하나씩 search_word로 호출하면서 매번 time.sleep(0.3)을 넣어 서버 부담을 줄였다.
각 응답의 data["channel"]["total"]로 검색어별 전체 결과 수를 출력
모든 응답의 item 리스트를 하나로 합친 뒤 collections.Counter의 most_common(3)으로 품사 빈도 상위 3개를 구함
pos가 없거나 빈 문자열이면 (미상)으로 처리했으며, 검색어가 모두 음식 이름이라 가장 흔한 품사는 명사로 나타난다.

(비빔밥에서 계속 오류가 발생하여 비빔밥을 제외하고 실행함)